# ru-tat-call — Colab ASR worker (шаг 4.1)

Поднимает HTTP-воркер на GPU и публикует его через **ngrok** или **cloudflared**.
Ноутбук на ноутбуке (`asr_server`) в шаге 4.2 будет слать PCM на этот URL.

**Runtime → Change runtime type → GPU (T4).**

Контракт:

`POST {public_url}/v1/transcribe`  
`{"audio_base64", "sample_rate": 16000, "encoding": "pcm_s16le"}` →  
`{"text", "language": "ru"|"tt"|"mixed"|"unknown", "is_final": true}`

Чекпоинты (из `context/asr.md`, не бенчмарк):
- wav2vec2: `anton-l/wav2vec2-large-xlsr-53-tatar`
- whisper: `openai/whisper-small` (RU + mixed baseline)

Ошибка воркера не должна ронять звонок: пустой `text` допустим.

In [ ]:
!nvidia-smi -L || echo "GPU не виден — Runtime → GPU"

In [ ]:
%pip install -q fastapi uvicorn pyngrok huggingface_hub transformers accelerate torch

## Код воркера из репозитория

Клонирует `develop` и запускает `project/apps/colab_asr/worker.py`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = os.environ.get("RU_TAT_CALL_REPO", "https://github.com/Dan1kMiniTogi/ru-tat-call.git")
BRANCH = os.environ.get("RU_TAT_CALL_BRANCH", "develop")
ROOT = Path("/content/ru-tat-call")
if not (ROOT / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only"])
WORKER = ROOT / "project" / "apps" / "colab_asr" / "worker.py"
assert WORKER.is_file(), WORKER
print("worker:", WORKER)

## Запуск HTTP на :8090

- `ASR_WORKER_BACKEND=wav2vec2` (татарский XLS-R) или `whisper`
- `dummy` — без скачивания модели (проверка туннеля)

Опционально `ASR_WORKER_TOKEN`: тогда ноутбук должен слать заголовок `X-Worker-Token` (шаг 4.2).

In [ ]:
import os, subprocess, time, urllib.request, sys

os.environ.setdefault("ASR_WORKER_BACKEND", "wav2vec2")
os.environ.setdefault("ASR_WORKER_MODEL", "anton-l/wav2vec2-large-xlsr-53-tatar")
# os.environ["ASR_WORKER_BACKEND"] = "whisper"
# os.environ["ASR_WORKER_MODEL"] = "openai/whisper-small"
os.environ.setdefault("ASR_WORKER_DEVICE", "cuda")

PORT = 8090
proc = subprocess.Popen(
    [sys.executable, str(WORKER), "--host", "127.0.0.1", "--port", str(PORT),
     "--backend", os.environ["ASR_WORKER_BACKEND"],
     "--model", os.environ.get("ASR_WORKER_MODEL", ""),
     "--device", os.environ["ASR_WORKER_DEVICE"]],
    cwd=str(WORKER.parent),
)
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("worker did not start")
print(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health").read().decode())

## Туннель A: ngrok (`pyngrok`)

В Colab: Secrets → `NGROK_AUTHTOKEN` (бесплатный токен с https://dashboard.ngrok.com ).
Скопируй **https URL** в `ASR_REMOTE_URL` на ноутбуке (шаг 4.2). Никому в чат токен не присылай.

In [ ]:
from pyngrok import ngrok

token = os.environ.get("NGROK_AUTHTOKEN", "")
try:
    from google.colab import userdata
    token = token or userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass
if not token:
    print("Нет NGROK_AUTHTOKEN — пропусти ячейку и используй cloudflared ниже")
else:
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(PORT, "http")
    print("ASR_REMOTE_URL=", tunnel.public_url)
    print("Проверка:", tunnel.public_url + "/health")

## Туннель B: cloudflared (без аккаунта ngrok)

В логе появится `https://….trycloudflare.com` — это и есть `ASR_REMOTE_URL` (без path).

In [ ]:
import os, stat, subprocess, urllib.request
from pathlib import Path

bin_path = Path("/tmp/cloudflared")
if not bin_path.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        bin_path,
    )
    bin_path.chmod(bin_path.stat().st_mode | stat.S_IEXEC)
print("Запусти в отдельной ячейке и оставь работать:")
print(f"  {bin_path} tunnel --url http://127.0.0.1:{PORT}")
# Раскомментируй, чтобы стартовать здесь (блокирует ячейку):
# subprocess.run([str(bin_path), "tunnel", "--url", f"http://127.0.0.1:{PORT}"])

## Что вписать на ноутбуке (после 4.2)

В `project/.env`:

```
ASR_ENGINE=remote
ASR_REMOTE_URL=https://ВАШ-ТУННЕЛЬ
```

Шаг 4.2 подключит `RemoteColabASREngine` к `POST /v1/transcribe`. Пока коннектор — заглушка, этот воркер можно проверить `curl` с Colab или с домашнего ПК на `/health` и `/v1/transcribe`.